# FinTech Onboarding Funnel Visualization

Connects to the local PostgreSQL database, aggregates funnel stage counts,
and renders an executive horizontal bar chart highlighting the KYC bottleneck.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
import seaborn as sns

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "fintech_growth",
    "user": "admin",
    "password": "password123",
}

FUNNEL_QUERY = """
WITH funnel_stages AS (
    SELECT
        e.user_id,
        MAX(CASE WHEN e.event_name = 'account_created' THEN 1 ELSE 0 END) AS stage_1_signup,
        MAX(CASE WHEN e.event_name = 'document_uploaded' THEN 1 ELSE 0 END) AS stage_2_doc_upload,
        MAX(CASE WHEN e.event_name = 'kyc_approved' THEN 1 ELSE 0 END) AS stage_3_kyc_approved,
        MAX(CASE WHEN e.event_name = 'first_deposit_initiated' THEN 1 ELSE 0 END) AS stage_4_first_deposit
    FROM user_events e
    GROUP BY e.user_id
)
SELECT
    SUM(stage_1_signup) AS account_created,
    SUM(stage_2_doc_upload) AS document_uploaded,
    SUM(stage_3_kyc_approved) AS kyc_approved,
    SUM(stage_4_first_deposit) AS first_deposit
FROM funnel_stages;
"""

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR
OUTPUT_PATH = PROJECT_ROOT / "dashboards" / "funnel_dashboard.png"

In [2]:
with psycopg2.connect(**DB_CONFIG) as conn:
    funnel_row = pd.read_sql(FUNNEL_QUERY, conn)

stage_labels = [
    "Account Created",
    "Document Uploaded",
    "KYC Approved (Bottleneck)",
    "First Deposit",
]
stage_counts = [
    int(funnel_row.loc[0, "account_created"]),
    int(funnel_row.loc[0, "document_uploaded"]),
    int(funnel_row.loc[0, "kyc_approved"]),
    int(funnel_row.loc[0, "first_deposit"]),
]

funnel_df = pd.DataFrame({"stage": stage_labels, "users": stage_counts})
funnel_df

/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_32810/3073581575.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  funnel_row = pd.read_sql(FUNNEL_QUERY, conn)


,stage,users
0,Account Created,1200
1,Document Uploaded,840
2,KYC Approved (Bottleneck),480
3,First Deposit,300


In [3]:
sns.set_theme(style="whitegrid", context="talk")

bar_colors = ["#5b9bd5", "#5b9bd5", "#d9534f", "#5b9bd5"]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    funnel_df["stage"],
    funnel_df["users"],
    color=bar_colors,
    edgecolor="white",
    height=0.65,
)

ax.invert_yaxis()
ax.set_xlabel("Users")
ax.set_ylabel("")
ax.set_title("Neobank Onboarding Funnel — Conversion Drop-off", pad=16, weight="bold")
ax.set_xlim(0, max(stage_counts) * 1.15)

for bar, count in zip(bars, stage_counts):
    ax.text(
        bar.get_width() + max(stage_counts) * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,}",
        va="center",
        ha="left",
        fontsize=12,
        fontweight="bold",
        color="#333333",
    )

sns.despine(left=True, bottom=False)
fig.tight_layout()

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PATH, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Dashboard saved to: {OUTPUT_PATH}")

Dashboard saved to: /Users/caue/Data Analyst portifolio/FinTech Growth & Revenue Funnel/dashboards/funnel_dashboard.png


/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_32810/341426016.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
